### Fahrradwerkstatt – Lösung Schritt für Schritt

*41. Bundeswettbewerb Informatik (2022), 1. Runde, Aufgabe 4*

Marc hat sich mit einer kleinen Fahrradwerkstatt selbstständig gemacht. Aufgrund des Fahrrad-Booms wird es zunehmend
schwierig, die ganzen Reparaturaufträge von Kunden abzuarbeiten.

Daher arbeitet Marc bereits jeden Tag von 9 bis 17 Uhr. Wenn er einen Auftrag beginnt, erledigt er ihn vollständig, bevor er
einen neuen Auftrag beginnt. Am Ende eines Arbeitstages unterbricht er gegebenenfalls den aktuellen Auftrag und nimmt ihn am
nächsten Arbeitstag wieder auf.

Bislang erledigt er die Aufträge einfach in der Reihenfolge des Eingangs. Damit sind manche Kunden unzufrieden: Kurze
Aufträge müssen teilweise sehr lange warten, weil die Werkstatt durch lange Aufträge blockiert ist. Marc hat sich bereits
ein anderes Verfahren überlegt: Wann immer er einen neuen Auftrag beginnt, wählt er den kürzesten unter den bereits
vorliegenden Aufträgen.

#### Aufgabe 4

1. Unterstütze Marc und schreibe ein Programm, das die beiden Verfahren simuliert. Dazu bekommt es eine Liste von
   Aufträgen. Für jeden Auftrag sind Eingangszeitpunkt und Bearbeitungsdauern in Minuten angegeben. Für die beiden Verfahren
   soll das Programm die durchschnittliche und die maximale Wartezeit der Aufträge (in Minuten) berechnen. Die Wartezeit
   eines Auftrags ist die Differenz zwischen dem Zeitpunkt seiner Fertigstellung und seinem Eingangszeitpunkt.
2. Begründe, warum auch bei dem zweiten Verfahren vermutlich nicht alle Kunden zufrieden sein werden.
3. Gibt es Verfahren, die du für besser geeignet hältst? Hierzu kannst du dir auch weitere Kennzahlen als nur die
   durchschnittliche und maximale Wartezeit der Aufträge überlegen. Verändere dein Programm entsprechend und beschreibe,
   inwiefern dein Verfahren deine Erwartungen erfüllt.

Wende dein Programm mindestens auf alle Beispiele an, die du auf den BWINF-Webseiten findest, und dokumentiere die Ergebnisse.

#### Beispieldaten (`beispieldaten/fahrradwerkstatt0.txt` bis `fahrradwerkstatt4.txt`)

Je Zeile ein Auftrag: Eingangszeitpunkt und Bearbeitungsdauer in Minuten. Der Eingangszeitpunkt zählt in Minuten ab Beginn des
ersten Tages um 0 Uhr (Tag 0), z. B. `1980` = 33 Stunden = Tag 1 um 9:00 Uhr. Es gibt keine README, die Datei `fahrradwerkstatt1.txt`
hat die meisten Aufträge (766).

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Simulation mit Ereignissen:** Wir lassen die Uhr nicht Minute für Minute laufen, sondern springen von Auftragsende zu Auftragsende
  (und, wenn die Werkstatt leer ist, bis zum nächsten Eingang).
- **Zeit umrechnen:** Die Nacht (17 bis 9 Uhr) zählt für Marcs Arbeit nicht. Rechnet man in *Arbeitsminuten*, ist die Simulation
  einfach, und nur am Ende rechnet man auf die Uhrzeit zurück (Division mit Rest, `divmod`).
- **Prioritäten:** Aus den wartenden Aufträgen wird immer der mit der höchsten Priorität genommen (kleinster Eingang, kürzeste Dauer, ...).
  Verfahren unterscheiden sich nur in dieser einen Auswahl.
- **Kennzahlen:** Mittelwert und Maximum allein verraten wenig. Zusätzlich helfen Anteile, Auslastung und Überhang.

**Nützliche Funktionen und Bibliotheken**
- `min(liste, key=...)` mit Tupeln: Python vergleicht Tupel Stelle für Stelle, `min` über `(dauer, eingang)` nimmt also die
  kürzeste Dauer und bei Gleichstand den früheren Eingang.
- `divmod(a, b)` liefert Ganzzahldivision und Rest in einem Schritt.
- Für Neugierige: `heapq` (Haufen) als Prioritätenliste.

## Die Idee

**Arbeitsminuten.** Ein Tag hat 1440 Minuten, Marc arbeitet davon nur 480 (von Minute 540 bis 1020 des Tages). Wir zählen deshalb eine
zweite Uhr, die nur während der Arbeitszeit läuft:

- Ein Zeitpunkt `t` (Minuten seit Tag 0, 0 Uhr) wird zu `tag * 480 + (Minuten innerhalb der Arbeitszeit des Tages)`. Kommt ein Auftrag
  nachts oder morgens vor 9 Uhr, zählt er ab dem nächsten Arbeitsbeginn, nach 17 Uhr ab dem nächsten Morgen.
- Auf dieser Arbeitsuhr kann Marc einen Auftrag der Dauer `d` einfach von `T` bis `T + d` bearbeiten. Dass er über Nacht pausiert,
  steckt in der Umrechnung zurück auf die Uhrzeit.

**Die Simulation.** Wir halten die Aufträge nach Eingang sortiert und merken uns die aktuelle Arbeitszeit `T` und die Liste der
*wartenden* Aufträge:

1. Alle Aufträge, die bis `T` eingegangen sind, kommen in die Wartenden.
2. Ist niemand da, springt `T` auf den Eingang des nächsten Auftrags (Marc wartet, die Werkstatt ist leer).
3. Sonst wählt das Verfahren einen Wartenden aus. Er wird bis zum Ende bearbeitet (`T` wächst um seine Dauer), danach notieren wir seine Wartezeit.

**Wichtig:** Marc kennt nur Aufträge, die schon eingegangen sind. Wer erst später kommt, darf bei der Auswahl keine Rolle spielen
(Marc ist nicht allwissend).

**Die Verfahren** unterscheiden sich nur in Schritt 3:

- **A:** frühester Eingang, bei Gleichstand der erste in der Datei,
- **B:** kürzeste Dauer, bei Gleichstand der frühere Eingang,
- **C** (unsere Verbesserung): wie B, aber wer länger als eine *Höchstwartezeit* gewartet hat, wird vorgezogen (dazu später mehr).

## Schritt 1: Einlesen

Wir speichern jeden Auftrag als Tupel `(eingang, dauer)`.

In [ ]:
def lies_auftraege(dateiname):
    auftraege = []
    with open(dateiname, encoding='utf-8-sig') as f:
        for zeile in f:
            if zeile.strip():
                eingang, dauer = zeile.split()
                auftraege.append((int(eingang), int(dauer)))
    return auftraege


auftraege = lies_auftraege('beispieldaten/fahrradwerkstatt0.txt')
print(len(auftraege), 'Aufträge, die ersten drei:', auftraege[:3])

114 Aufträge, die ersten drei: [(2389, 1325), (6430, 427), (10620, 1541)]


## Schritt 2: Zwischen Uhrzeit und Arbeitsminuten umrechnen

`arbeitszeit(t)` zählt, wie viele Arbeitsminuten bis zum Zeitpunkt `t` vergangen sind: `tag` volle Tage zu je 480 Minuten, dazu der Teil
des angefangenen Tages, der in die Arbeitszeit fällt (zwischen Minute 540 und 1020, davor 0, danach 480).

`uhrzeit(w)` ist die Umkehrung: Nach `w` Arbeitsminuten ist man im Tag `(w - 1) // 480` (das `- 1` sorgt dafür, dass genau 17:00 Uhr noch
zum selben Tag gehört), und dort sind `w - tag * 480` Minuten ab 9 Uhr vergangen.

In [ ]:
BEGINN = 9 * 60          # Arbeitsbeginn (Minute des Tages)
ENDE = 17 * 60           # Arbeitsende
TAGESARBEIT = ENDE - BEGINN     # 480 Arbeitsminuten pro Tag


def arbeitszeit(t):
    tag, rest = divmod(t, 24 * 60)
    return tag * TAGESARBEIT + min(max(rest - BEGINN, 0), TAGESARBEIT)


def uhrzeit(w):
    if w == 0:
        return BEGINN
    tag = (w - 1) // TAGESARBEIT
    return tag * 24 * 60 + BEGINN + (w - tag * TAGESARBEIT)


def als_text(t):
    tag, rest = divmod(t, 24 * 60)
    return f'Tag {tag}, {rest // 60}:{rest % 60:02d} Uhr'


print(als_text(1980), '->', arbeitszeit(1980), 'Arbeitsminuten')      # genau um 9 Uhr an Tag 1
print(als_text(2389), '->', arbeitszeit(2389), 'Arbeitsminuten')
print(als_text(2000 + 960), '->', arbeitszeit(2000 + 960), 'Arbeitsminuten')   # nachts: wie 9 Uhr am nächsten Tag
print(als_text(uhrzeit(480)), '/', als_text(uhrzeit(481)))               # Ende von Tag 0 / 1 Minute am nächsten Tag

Tag 1, 9:00 Uhr -> 480 Arbeitsminuten
Tag 1, 15:49 Uhr -> 889 Arbeitsminuten
Tag 2, 1:20 Uhr -> 960 Arbeitsminuten
Tag 0, 17:00 Uhr / Tag 1, 9:01 Uhr


## Schritt 3: Die Auswahl der Verfahren

Die Funktion `waehle` bekommt die Liste der wartenden Aufträge und gibt den ausgewählten zurück. Ein wartender Auftrag ist ein Tupel
`(eingang_arbeit, dauer, eingang)` – der Eingang als Arbeitsminute (zum Rechnen) und als Uhrzeit (für die Wartezeit). Das Verfahren
geben wir als Buchstabe an (keine Funktion als Argument). Die Liste `wartend` bleibt in der Reihenfolge des Eingangs, daher ist für A der
erste Eintrag der richtige.

Für Verfahren C brauchen wir noch die aktuelle Uhrzeit `jetzt` und die `grenze` (Höchstwartezeit in Minuten): Zuerst sehen wir nach, ob
jemand schon länger als `grenze` Minuten wartet. Wenn ja, bekommt der älteste unter ihnen den Zuschlag, sonst gilt Verfahren B.

In [ ]:
def waehle(wartend, verfahren, jetzt=0, grenze=0):
    if verfahren == 'A':
        return wartend[0]
    if verfahren == 'C':
        zu_lange = [a for a in wartend if jetzt - a[2] > grenze]
        if zu_lange:
            return zu_lange[0]          # der älteste (Liste ist nach Eingang geordnet)
    return min(wartend, key=lambda a: (a[1], a[2]))      # Verfahren B (und C ohne Überschreitung)

Die Schreibweise `key=lambda a: (a[1], a[2])` bedeutet: „vergleiche die Aufträge nach dem Tupel `(Dauer, Eingang)`“. Das ist nur eine
kompakte Schreibweise für „kürzeste Dauer, bei Gleichstand früherer Eingang“.

## Schritt 4: Die Simulation

Pro abgearbeitetem Auftrag merken wir uns: Eingang, Dauer und den Zeitpunkt der Fertigstellung (Uhrzeit). Dazu brauchen wir die Arbeitsuhr `T`:

In [ ]:
def simuliere(auftraege, verfahren, grenze=0):
    """Gibt eine Liste (eingang, dauer, fertig) in der Reihenfolge der Bearbeitung zurück."""
    offen = sorted((arbeitszeit(e), d, e) for e, d in auftraege)    # (Eingang in Arbeitsminuten, Dauer, Eingang)
    naechster = 0               # Index des nächsten noch nicht eingegangenen Auftrags
    wartend = []
    T = 0                       # Arbeitsuhr
    ergebnis = []
    while len(ergebnis) < len(offen):
        # Alle Aufträge, die bis T eingegangen sind, kommen in die Warteschlange
        while naechster < len(offen) and offen[naechster][0] <= T:
            wartend.append(offen[naechster])
            naechster += 1
        if not wartend:
            T = offen[naechster][0]         # Werkstatt leer: bis zum nächsten Eingang warten
            continue
        auftrag = waehle(wartend, verfahren, uhrzeit(T), grenze)
        wartend.remove(auftrag)
        T += auftrag[1]
        ergebnis.append((auftrag[2], auftrag[1], uhrzeit(T)))
    return ergebnis


for eingang, dauer, fertig in simuliere(auftraege, 'A')[:3]:
    print('Eingang', als_text(eingang), '| Dauer', dauer, '| fertig', als_text(fertig))

Eingang Tag 1, 15:49 Uhr | Dauer 1325 | fertig Tag 4, 13:54 Uhr
Eingang Tag 4, 11:10 Uhr | Dauer 427 | fertig Tag 5, 13:01 Uhr
Eingang Tag 7, 9:00 Uhr | Dauer 1541 | fertig Tag 10, 10:41 Uhr


## Schritt 5: Kennzahlen

Die **Wartezeit** eines Auftrags ist Fertigstellung minus Eingang (in Minuten, die Nacht zählt mit). `kennzahlen` gibt die mittlere und die
größte Wartezeit zurück. Weil die Nacht die Zahlen stark aufbläht, berechnen wir auch die Wartezeit **ohne Totzeit**, also nur in
Arbeitsminuten (`arbeitszeit(fertig) - arbeitszeit(eingang)`). So sieht man, wie lange Marc rechnerisch „hinterherhinkt“.

In [ ]:
def kennzahlen(ergebnis):
    """Mittel und Maximum der Wartezeit, mit und ohne Totzeit (Nacht)."""
    mit = [fertig - eingang for eingang, dauer, fertig in ergebnis]
    ohne = [arbeitszeit(fertig) - arbeitszeit(eingang) for eingang, dauer, fertig in ergebnis]
    return sum(mit) / len(mit), max(mit), sum(ohne) / len(ohne), max(ohne)


for verfahren in 'AB':
    mittel, groesste, mittel_o, groesste_o = kennzahlen(simuliere(auftraege, verfahren))
    print(f'Verfahren {verfahren}: mittlere Wartezeit {mittel:.0f} min, größte {groesste} min '
          f'(ohne Totzeit: {mittel_o:.0f} und {groesste_o})')

Verfahren A: mittlere Wartezeit 32743 min, größte 68771 min (ohne Totzeit: 11000 und 22943)
Verfahren B: mittlere Wartezeit 16981 min, größte 188734 min (ohne Totzeit: 5764 und 62974)


## Alle Beispiele (Aufgabe 4.1)

Wir lassen A und B auf allen fünf Dateien laufen. Zum Vergleich stehen rechts die Werte der Lösungshinweise (mit Totzeit, gerundet).

In [ ]:
HINWEISE = {      # (mittel A, max A, mittel B, max B) aus den Lösungshinweisen
    0: (32754, 68771, 16986, 188734),
    1: (63536, 128321, 11908, 433563),
    2: (51200, 110973, 14862, 327087),
    3: (30029, 60821, 17246, 382016),
    4: (74428, 167059, 42201, 363155),
}
print('Datei  Aufträge |       A: Mittel    Max |       B: Mittel     Max | Hinweise: A Mittel/Max, B Mittel/Max')
for nr in range(5):
    liste = lies_auftraege(f'beispieldaten/fahrradwerkstatt{nr}.txt')
    a = kennzahlen(simuliere(liste, 'A'))
    b = kennzahlen(simuliere(liste, 'B'))
    print(f'{nr:>5}  {len(liste):>8} | {a[0]:>12.0f} {a[1]:>6} | {b[0]:>12.0f} {b[1]:>7} |', HINWEISE[nr])

Datei  Aufträge |       A: Mittel    Max |       B: Mittel     Max | Hinweise: A Mittel/Max, B Mittel/Max
    0       114 |        32743  68771 |        16981  188734 | (32754, 68771, 16986, 188734)
    1       766 |        63531 128321 |        11884  433563 | (63536, 128321, 11908, 433563)
    2       186 |        51194 110973 |        14806  327087 | (51200, 110973, 14862, 327087)
    3       220 |        30023  60821 |        17243  382016 | (30029, 60821, 17246, 382016)
    4        90 |        74428 167059 |        42201  363155 | (74428, 167059, 42201, 363155)


**Vergleich mit den Lösungshinweisen.** Die größten Wartezeiten stimmen genau überein, die mittleren Wartezeiten weichen um höchstens
etwa 0,1 % ab (die Hinweise rechnen offenbar an irgendeiner Stelle um ein paar Minuten anders). Für die Wartezeiten ohne Totzeit nennen
die Hinweise ebenfalls Werte, die um etwa 2 % von unseren abweichen (Beispiel `fahrradwerkstatt0.txt`, Verfahren A: Hinweise 10757 und
23475, bei uns 11000 und 22943). Wie dort Aufträge behandelt werden, die außerhalb der Arbeitszeit eingehen, ist nicht beschrieben.
Wir bleiben bei unserer Definition (siehe Schritt 5) und lassen diese Abweichung stehen.

**Beobachtungen**

- Die **mittlere** Wartezeit ist bei B in allen Beispielen deutlich kleiner als bei A, bei `fahrradwerkstatt1.txt` sogar um den Faktor 5 bis 6.
- Die **größte** Wartezeit ist bei B dagegen deutlich größer als bei A, in `fahrradwerkstatt3.txt` mehr als das Sechsfache.

## Aufgabe 4.2: Warum sind auch bei B nicht alle Kunden zufrieden?

Verfahren B bevorzugt kurze Aufträge. Solange Marc ausgelastet ist, wird ein langer Auftrag immer wieder von kürzeren überholt, die erst
danach eingegangen sind. Im Extremfall kommt ein langer Auftrag nie an die Reihe, solange neue kurze Aufträge nachkommen
(man nennt das **Verhungern**). Die Zahlen zeigen es: Bei `fahrradwerkstatt1.txt` wartet im schlimmsten Fall ein Kunde bei A rund
128 000 Minuten, bei B rund 434 000 Minuten, mehr als das Dreifache. Außerdem kann der Kunde mit dem langen Auftrag die Wartezeit
nicht abschätzen, denn sie hängt davon ab, wer später noch kommt. Bei A weiß ein Kunde dagegen, dass vor ihm nur die schon vorhandenen
Aufträge dran sind. Die Mittelwerte allein täuschen also: Sehr viele zufriedene kurze Aufträge ziehen den Durchschnitt nach unten,
einige wenige sehr unzufriedene Kunden gehen darin unter.

Der Unterschied wird deutlicher, wenn man nicht nur Mittel und Maximum betrachtet, sondern wissen will, **wie viele Kunden wie lange
warten**. Wir zählen, welcher Anteil der Aufträge länger als 30 Tage (43 200 Minuten) wartet, und bestimmen die Wartezeit, die 90 % der Aufträge
nicht überschreiten (das 90-%-Quantil).

In [ ]:
def anteil_ueber(ergebnis, grenze):
    wartezeiten = [fertig - eingang for eingang, dauer, fertig in ergebnis]
    return sum(1 for w in wartezeiten if w > grenze) / len(wartezeiten)


def quantil(ergebnis, p):
    wartezeiten = sorted(fertig - eingang for eingang, dauer, fertig in ergebnis)
    return wartezeiten[int(p * (len(wartezeiten) - 1))]


print('Datei | Anteil > 30 Tage: A, B | 90-%-Wartezeit: A, B')
for nr in range(5):
    liste = lies_auftraege(f'beispieldaten/fahrradwerkstatt{nr}.txt')
    a = simuliere(liste, 'A')
    b = simuliere(liste, 'B')
    print(f'{nr:>5} | {anteil_ueber(a, 43200):>8.0%} {anteil_ueber(b, 43200):>8.0%} | {quantil(a, 0.9):>13} {quantil(b, 0.9):>8}')

Datei | Anteil > 30 Tage: A, B | 90-%-Wartezeit: A, B
    0 |      38%      10% |         57647    30475
    1 |      64%       5% |        121124     8713
    2 |      62%       6% |         84806    27358
    3 |      28%       6% |         52188    27522
    4 |      71%      20% |        144191   129722


Bei B bleiben 90 % der Kunden weit unter den Wartezeiten von A (bei `fahrradwerkstatt1.txt` unter 9 000 statt 121 000 Minuten), dafür
warten die übrigen 5 bis 20 % (je nach Datei) länger als 30 Tage. Das sind genau die Kunden mit den langen Aufträgen, die B zurückstellt.

## Aufgabe 4.3: Ein besseres Verfahren (C) und weitere Kennzahlen

**Verfahren C: kürzester Auftrag zuerst, aber mit Höchstwartezeit.** Wir übernehmen B und fügen eine Sicherung ein: Hat ein Auftrag
länger als `grenze` Minuten gewartet, wird er allen anderen vorgezogen (der älteste zuerst). Das haben wir in `waehle` schon eingebaut.
Die Frage ist die Höhe der Grenze: Sehr große Grenze = Verfahren B, sehr kleine Grenze = fast Verfahren A. Wir probieren mehrere Werte aus
und sehen uns Mittel und Maximum an (in Minuten, mit Totzeit):

In [ ]:
GRENZEN = [10000, 20000, 50000, 100000]
for nr in range(5):
    liste = lies_auftraege(f'beispieldaten/fahrradwerkstatt{nr}.txt')
    zeile = f'Datei {nr}:'
    for grenze in GRENZEN:
        mittel, groesste = kennzahlen(simuliere(liste, 'C', grenze))[:2]
        zeile += f'  Grenze {grenze}: {mittel:.0f} / {groesste} |'
    print(zeile)

Datei 0:  Grenze 10000: 32576 / 68771 |  Grenze 20000: 30413 / 68771 |  Grenze 50000: 22460 / 85199 |  Grenze 100000: 17966 / 122403 |
Datei 1:  Grenze 10000: 63264 / 128321 |  Grenze 20000: 62800 / 128321 |  Grenze 50000: 56481 / 128321 |  Grenze 100000: 41731 / 128321 |
Datei 2:  Grenze 10000: 51093 / 110973 |  Grenze 20000: 49782 / 110973 |  Grenze 50000: 36660 / 110973 |  Grenze 100000: 19751 / 129870 |
Datei 3:  Grenze 10000: 29705 / 60821 |  Grenze 20000: 29118 / 60821 |  Grenze 50000: 20421 / 67727 |  Grenze 100000: 18061 / 111305 |
Datei 4:  Grenze 10000: 74158 / 167059 |  Grenze 20000: 74127 / 167059 |  Grenze 50000: 69717 / 167059 |  Grenze 100000: 53615 / 171549 |


Die Tabelle zeigt den Zielkonflikt (Mittel / Maximum): Bei kleiner Grenze (10000, 20000) verhält sich C fast wie A, die größte Wartezeit
ist sogar dieselbe, nur der Mittelwert sinkt etwas. Bei großer Grenze (100000) nähert sich C dem Verfahren B im Mittelwert, die größte
Wartezeit liegt aber nur wenig über der von A (z. B. Datei 3: 111305 statt 382016 bei B, mittlere Wartezeit 18061 statt 30023 bei A).
Wo man die Grenze setzt, ist eine Frage dessen, was Marc seinen Kunden zumuten will. Für den Vergleich nehmen wir `grenze = 50000`
(etwa 35 Tage, der Wert, den auch die Lösungshinweise verwenden) und stellen A, B und C nebeneinander:

In [ ]:
print('Datei | Mittel A / B / C | Maximum A / B / C')
for nr in range(5):
    liste = lies_auftraege(f'beispieldaten/fahrradwerkstatt{nr}.txt')
    k = [kennzahlen(simuliere(liste, v, 50000)) for v in 'ABC']
    print(f'{nr:>5} | {k[0][0]:>6.0f} {k[1][0]:>6.0f} {k[2][0]:>6.0f} | {k[0][1]:>6} {k[1][1]:>6} {k[2][1]:>6}')

Datei | Mittel A / B / C | Maximum A / B / C
    0 |  32743  16981  22460 |  68771 188734  85199
    1 |  63531  11884  56481 | 128321 433563 128321
    2 |  51194  14806  36660 | 110973 327087 110973
    3 |  30023  17243  20421 |  60821 382016  67727
    4 |  74428  42201  69717 | 167059 363155 167059


**Auswertung.** Die Hinweise nennen für C andere Zahlen (z. B. bei `fahrradwerkstatt0.txt` mittlere Wartezeit 22344 und größte 89663, bei uns
22460 und 85199; bei `fahrradwerkstatt1.txt` 35761 und 331601, bei uns 56481 und 128321). Die genaue Regel und der Zeitpunkt der Prüfung
sind dort nicht beschrieben, wir erhalten also teils andere Werte. Das Muster ist aber dasselbe: C liegt beim Mittelwert zwischen A und B
und bei der größten Wartezeit deutlich unter B. Bei unserer Grenze 50000 bleibt die größte Wartezeit in den Dateien 1, 2 und 4 genauso hoch wie bei
A, weil dort die Grenze fast immer überschritten ist und C dann wie A arbeitet. Der Mittelwert ist trotzdem etwas besser als bei A, bei
Datei 2 deutlich (36660 statt 51194). Unsere Erwartung (guter Mittelwert, aber kein Kunde wartet extrem lange) erfüllt C bei einer
größeren Grenze (100000) am besten. Welche Grenze passt, hängt von den Daten ab.

### Weitere Kennzahlen

- **Verzögerung** = Fertigstellung − Eingang − Dauer: Was der Kunde zusätzlich zur angekündigten Reparaturzeit wartet.
- **Auslastung** = genutzte Arbeitszeit / vergangene Arbeitszeit bis zum letzten Auftragsende. Sie ist kleiner als 100 %, wenn Marc
  warten muss, weil niemand da ist. Sie hängt nicht vom Verfahren ab, solange Marc nie freiwillig wartet.
- **Überhang** = Arbeitsende − Eingang des letzten Auftrags: Wie lange Marc nach dem letzten Eingang noch arbeitet.

In [ ]:
def verzoegerung(ergebnis):
    v = [fertig - eingang - dauer for eingang, dauer, fertig in ergebnis]
    return sum(v) / len(v)


def auslastung(ergebnis):
    genutzt = sum(dauer for eingang, dauer, fertig in ergebnis)
    vergangen = arbeitszeit(max(fertig for eingang, dauer, fertig in ergebnis))
    return genutzt / vergangen


def ueberhang(ergebnis):
    return max(fertig for eingang, dauer, fertig in ergebnis) - max(eingang for eingang, dauer, fertig in ergebnis)


print('Datei | mittlere Verzögerung A / B / C | Auslastung A / B | Überhang A / B / C')
for nr in range(5):
    liste = lies_auftraege(f'beispieldaten/fahrradwerkstatt{nr}.txt')
    e = [simuliere(liste, v, 50000) for v in 'ABC']
    print(f'{nr:>5} | {verzoegerung(e[0]):>7.0f} {verzoegerung(e[1]):>7.0f} {verzoegerung(e[2]):>7.0f} |'
          f' {auslastung(e[0]):>6.1%} {auslastung(e[1]):>6.1%} | {ueberhang(e[0]):>6} {ueberhang(e[1]):>6} {ueberhang(e[2]):>6}')

Datei | mittlere Verzögerung A / B / C | Auslastung A / B | Überhang A / B / C
    0 |   31192   15430   20909 |  91.0%  91.0% |  57573  57573  57573
    1 |   63300   11653   56250 |  86.7%  86.7% |  86523  86523  86523
    2 |   50250   13862   35716 |  95.3%  95.3% |  27446  27446  27446
    3 |   29225   16445   19623 |  95.0%  95.0% |  28839  28839  28839
    4 |   72433   40207   67723 |  91.5%  91.5% |  65099  65099  65099


Die Auslastung ist bei A und B gleich (87 bis 95 %), denn Marc arbeitet immer, wenn ein Auftrag da ist. Die Hinweise nennen dafür
„Effektivität“ 91,1 %, 86,8 %, 95,4 %, 95,0 % und 91,5 %, das passt zu unseren Werten. Auch der Überhang ist bei uns für alle Verfahren
gleich (Datei 0: 57573 Minuten), denn die Summe der Arbeit ändert sich durch die Reihenfolge nicht. Die Hinweise nennen 56234 für A und
51467 für B; wie sie rechnen, ist nicht beschrieben, wir bleiben bei unserer einfachen Definition. Die Verzögerung zeigt wie die
Wartezeit: B ist im Mittel viel besser als A, C liegt dazwischen.

**Fazit.** Verfahren B ist für den Durchschnitt am besten, aber ungerecht gegenüber langen Aufträgen. Verfahren A ist fair, aber im Mittel
langsam. C mit einer Höchstwartezeit liegt dazwischen. Welches „besser“ ist, hängt davon ab, wie viel Marc an der Zufriedenheit
*aller* Kunden liegt. Je kleiner die Grenze, desto mehr verhält sich C wie A.

## Für Neugierige: Prioritätenliste mit `heapq`

Unser `min` über die wartende Liste braucht für jede Auswahl so viele Schritte, wie Aufträge warten. Bei der Datei mit 766 Aufträgen ist das
noch unproblematisch. Bei sehr vielen Aufträgen nimmt man einen **Haufen** (englisch *heap*), der das kleinste Element schnell
herausgibt. Das Modul `heapq` bietet ihn für Listen: `heappush` fügt ein, `heappop` entnimmt das kleinste Element (nach Tupelreihenfolge).
Die Reihenfolge der Tupel `(dauer, eingang)` ist genau die Priorität von Verfahren B. Für B (und analog A) ändert sich die Priorität
eines Auftrags nie, daher geht das. Bei C ändert sie sich mit der Zeit, das ist umständlicher.

In [ ]:
import heapq


def simuliere_b_mit_haufen(auftraege):
    offen = sorted((arbeitszeit(e), d, e) for e, d in auftraege)
    naechster = 0
    haufen = []
    T = 0
    ergebnis = []
    while len(ergebnis) < len(offen):
        while naechster < len(offen) and offen[naechster][0] <= T:
            ein_arbeit, dauer, eingang = offen[naechster]
            heapq.heappush(haufen, (dauer, eingang))
            naechster += 1
        if not haufen:
            T = offen[naechster][0]
            continue
        dauer, eingang = heapq.heappop(haufen)
        T += dauer
        ergebnis.append((eingang, dauer, uhrzeit(T)))
    return ergebnis


liste = lies_auftraege('beispieldaten/fahrradwerkstatt1.txt')
print(simuliere_b_mit_haufen(liste) == simuliere(liste, 'B'))

True


Die Ergebnisse sind identisch.

**MiniZinc?** Hier wird nichts „gesucht“, sondern ein Ablauf Schritt für Schritt nachgespielt, in dem jede Entscheidung nur von den bisher
eingegangenen Aufträgen abhängt. Das ist mit einer einfachen Schleife natürlicher als mit einem Constraint-Modell, daher gibt es in diesem
Notebook keinen MiniZinc-Abschnitt.